In [1]:
import pandas as pd
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from nltk.tokenize import word_tokenize
from itertools import combinations
import torch
import numpy as np
import random
import os


import sys
sys.path.append('./tools')
from Matave import Matave

In [2]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

In [3]:
k_range = list(range(3, 20))
top_n = 10

synthetic_dataset = pd.read_csv('../dataSynthetic/makeOneBigFile/dataSyntheticAll.csv')
all_texts = synthetic_dataset['report'].values.tolist()
unmet_texts = synthetic_dataset[synthetic_dataset['needs'] == 'unmet']['report'].values.tolist()
met_texts = synthetic_dataset[synthetic_dataset['needs'] == 'met']['report'].values.tolist()

In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def matave_compute(texts):
    # Run matave. 
    matave = Matave(texts)
    matave.fit(k_range = k_range)
    cluster_topics = [topic.split() for topic in matave.top_topic_words.values()]
    cluster_topics = [topic[:top_n] for topic in cluster_topics]

    # Prepare components for evaluation.
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    # Evaluate using metrics.
    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}\nDiversity: {diversity}\nRedundancy: {redundancy}")
    
    matave.visualize()

    return matave, cluster_topics, {'coherence': coherence, 'diversity': diversity, 'redundancy': redundancy}

In [6]:
all_matave, all_cluster_topics, all_metrics = matave_compute(all_texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/91 [00:00<?, ?it/s]

/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Coherence: 0.3927067006208963
Diversity: 0.68
Redundancy: 0.9066666666666666


In [7]:
met_matave, met_cluster_topics, met_metrics = matave_compute(met_texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

Coherence: 0.3886715109001132
Diversity: 0.72
Redundancy: 0.86


In [8]:
unmet_matave, unmet_cluster_topics, unmet_metrics = matave_compute(unmet_texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/45 [00:00<?, ?it/s]

/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Coherence: 0.3754886050660042
Diversity: 0.8888888888888888
Redundancy: 0.9694444444444444
